In [1]:
import pandas as pd
import requests
from datetime import datetime
from typing import Dict, Optional, Tuple

In [2]:
INPUT_FILE = "vona_3gunung_knn_imputed.csv"
OUTPUT_FILE = "vona_3gunung_knn_with_wind.csv"

TIMESTAMP_COLUMN = "timestamp"
VOLCANO_COLUMN = "volcano_filter"
LAT_COLUMN = "lat"
LON_COLUMN = "long"
WIND_SPEED_COLUMN = "kec_angin_km_jam"
WIND_DIRECTION_COLUMN = "arah_angin_deg"

API_TIMEOUT_SECONDS = 15

In [3]:
def parse_timestamp(timestamp_str: str) -> Tuple[Optional[str], Optional[int]]:
    try:
        dt = datetime.strptime(timestamp_str, "%Y-%m-%d %H:%M:%S %Z")
        return dt.strftime("%Y-%m-%d"), dt.hour
    except Exception as error:
        print(f"Error parsing timestamp {timestamp_str}: {error}")
        return None, None

In [4]:
def get_wind_data(lat: float, lon: float, date_str: str, hour: int) -> Tuple[Optional[float], Optional[float]]:
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_str,
        "end_date": date_str,
        "hourly": "wind_speed_10m,wind_direction_10m",
        "wind_speed_unit": "kmh",
        "timezone": "UTC",
    }

    try:
        response = requests.get(url, params=params, timeout=API_TIMEOUT_SECONDS)
        if response.status_code != 200:
            print(f"API error {response.status_code}: {response.text}")
            return None, None

        payload = response.json()
        hourly_data = payload.get("hourly")
        if not hourly_data:
            print(f"No hourly data available for {date_str}")
            return None, None

        wind_speed = None
        wind_direction = None

        speeds = hourly_data.get("wind_speed_10m", [])
        if hour < len(speeds) and speeds[hour] is not None:
            wind_speed = round(speeds[hour], 2)

        directions = hourly_data.get("wind_direction_10m", [])
        if hour < len(directions) and directions[hour] is not None:
            wind_direction = round(directions[hour], 1)

        return wind_speed, wind_direction
    except Exception as error:
        print(f"Request failed: {error}")
        return None, None

In [5]:
def is_empty_value(value) -> bool:
    return pd.isna(value) or value == ""


def is_valid_coordinate(lat, lon) -> bool:
    if is_empty_value(lat) or is_empty_value(lon):
        return False

    try:
        lat_val = float(lat)
        lon_val = float(lon)
    except (TypeError, ValueError):
        return False

    return -90 <= lat_val <= 90 and -180 <= lon_val <= 180


def build_volcano_coordinate_map(dataframe: pd.DataFrame) -> Dict[str, Tuple[float, float]]:
    coordinate_map: Dict[str, Tuple[float, float]] = {}

    for _, row in dataframe.iterrows():
        volcano_name = row.get(VOLCANO_COLUMN)
        lat = row.get(LAT_COLUMN)
        lon = row.get(LON_COLUMN)

        if volcano_name in coordinate_map:
            continue

        if is_valid_coordinate(lat, lon):
            coordinate_map[volcano_name] = (float(lat), float(lon))

    return coordinate_map


def resolve_row_coordinates(row: pd.Series, coordinate_map: Dict[str, Tuple[float, float]]) -> Tuple[Optional[float], Optional[float], str]:
    lat = row.get(LAT_COLUMN)
    lon = row.get(LON_COLUMN)

    if is_valid_coordinate(lat, lon):
        return float(lat), float(lon), "row"

    volcano_name = row.get(VOLCANO_COLUMN)
    fallback_coords = coordinate_map.get(volcano_name)
    if fallback_coords:
        return fallback_coords[0], fallback_coords[1], "fallback_volcano"

    return None, None, "missing"


df = pd.read_csv(INPUT_FILE)
volcano_coordinate_map = build_volcano_coordinate_map(df)

print(f"Total data: {len(df)} rows")
print(f"Input file: {INPUT_FILE}")
print(f"Known volcano fallback coords: {len(volcano_coordinate_map)}")
print("-" * 60)

Total data: 98 rows
Input file: vona_3gunung_knn_imputed.csv
Known volcano fallback coords: 3
------------------------------------------------------------


In [6]:
def process_missing_wind_data(dataframe: pd.DataFrame, coordinate_map: Dict[str, Tuple[float, float]]) -> Dict[str, int]:
    print("=" * 60)
    print("Processing wind data for empty values...")
    print("=" * 60)

    stats = {
        "speed_filled": 0,
        "direction_filled": 0,
        "api_calls": 0,
        "invalid_timestamp": 0,
        "missing_coordinates": 0,
        "fallback_coordinate_used": 0,
    }

    total_rows = len(dataframe)

    for index, row in dataframe.iterrows():
        current_speed = row[WIND_SPEED_COLUMN]
        current_direction = row[WIND_DIRECTION_COLUMN]

        speed_empty = is_empty_value(current_speed)
        direction_empty = is_empty_value(current_direction)

        if not (speed_empty or direction_empty):
            print(
                f"Row {index + 1}: Already complete (Speed: {current_speed} km/h, Direction: {current_direction} deg)"
            )
            continue

        timestamp_str = row[TIMESTAMP_COLUMN]
        date_str, hour = parse_timestamp(timestamp_str)
        if date_str is None or hour is None:
            print(f"Skipping row {index + 1}: Invalid timestamp")
            stats["invalid_timestamp"] += 1
            print()
            continue

        lat, lon, source = resolve_row_coordinates(row, coordinate_map)
        if lat is None or lon is None:
            print(f"Skipping row {index + 1}: Missing valid coordinates (no row/fallback)")
            stats["missing_coordinates"] += 1
            print()
            continue

        if source == "fallback_volcano":
            stats["fallback_coordinate_used"] += 1

        print(f"Processing row {index + 1}/{total_rows}: {timestamp_str} | lat={lat}, lon={lon} ({source})")
        wind_speed, wind_direction = get_wind_data(lat, lon, date_str, hour)

        if speed_empty and wind_speed is not None:
            dataframe.at[index, WIND_SPEED_COLUMN] = wind_speed
            stats["speed_filled"] += 1
            print(f"  Wind speed: {wind_speed} km/h")

        if direction_empty and wind_direction is not None:
            dataframe.at[index, WIND_DIRECTION_COLUMN] = wind_direction
            stats["direction_filled"] += 1
            print(f"  Wind direction: {wind_direction} deg")

        stats["api_calls"] += 1
        print()

    print("=" * 60)
    print("Processing summary:")
    print(f"Wind speeds filled: {stats['speed_filled']}")
    print(f"Wind directions filled: {stats['direction_filled']}")
    print(f"API calls made: {stats['api_calls']}")
    print(f"Rows skipped (invalid timestamp): {stats['invalid_timestamp']}")
    print(f"Rows skipped (missing coordinates): {stats['missing_coordinates']}")
    print(f"Fallback coordinates used: {stats['fallback_coordinate_used']}")
    print("=" * 60)

    return stats


stats = process_missing_wind_data(df, volcano_coordinate_map)

Processing wind data for empty values...
Processing row 1/98: 2020-12-28 02:17:00 UTC | lat=-7.941944444, lon=112.95 (row)
  Wind speed: 7.1 km/h
  Wind direction: 240 deg

Processing row 2/98: 2016-10-19 01:12:00 UTC | lat=-7.941944444, lon=112.95 (row)
  Wind speed: 4.3 km/h
  Wind direction: 114 deg

Processing row 3/98: 2016-07-12 02:09:00 UTC | lat=-7.941944444, lon=112.95 (row)
  Wind speed: 5.9 km/h
  Wind direction: 11 deg

Processing row 4/98: 2016-05-24 05:31:00 UTC | lat=-7.941944444, lon=112.95 (row)
  Wind speed: 3.7 km/h
  Wind direction: 101 deg

Processing row 5/98: 2016-04-03 04:52:00 UTC | lat=-7.941944444, lon=112.95 (row)
  Wind speed: 6.1 km/h
  Wind direction: 45 deg

Processing row 6/98: 2016-04-03 04:21:00 UTC | lat=-7.941944444, lon=112.95 (row)
  Wind speed: 6.1 km/h
  Wind direction: 45 deg

Processing row 7/98: 2016-04-03 04:07:00 UTC | lat=-7.941944444, lon=112.95 (row)
  Wind speed: 6.1 km/h
  Wind direction: 45 deg

Processing row 8/98: 2016-04-02 04:45:0

In [7]:
df.to_csv(OUTPUT_FILE, index=False)
print("=" * 60)
print(f"Data saved to: {OUTPUT_FILE}")
print(f"Total rows: {len(df)}")
print("=" * 60)

Data saved to: vona_3gunung_knn_with_wind.csv
Total rows: 98


In [8]:
print("Data preview:")
print(df[[TIMESTAMP_COLUMN, VOLCANO_COLUMN, LAT_COLUMN, LON_COLUMN, WIND_SPEED_COLUMN, WIND_DIRECTION_COLUMN]].head(10))

Data preview:
                 timestamp volcano_filter       lat    long  kec_angin_km_jam  \
0  2020-12-28 02:17:00 UTC          Bromo -7.941944  112.95               7.1   
1  2016-10-19 01:12:00 UTC          Bromo -7.941944  112.95               4.3   
2  2016-07-12 02:09:00 UTC          Bromo -7.941944  112.95               5.9   
3  2016-05-24 05:31:00 UTC          Bromo -7.941944  112.95               3.7   
4  2016-04-03 04:52:00 UTC          Bromo -7.941944  112.95               6.1   
5  2016-04-03 04:21:00 UTC          Bromo -7.941944  112.95               6.1   
6  2016-04-03 04:07:00 UTC          Bromo -7.941944  112.95               6.1   
7  2016-04-02 04:45:00 UTC          Bromo -7.941944  112.95               4.6   
8  2016-04-02 04:19:00 UTC          Bromo -7.941944  112.95               4.6   
9  2019-07-19 11:52:00 UTC          Bromo -7.941944  112.95               2.9   

   arah_angin_deg  
0           240.0  
1           114.0  
2            11.0  
3           10